In [1]:
from pathlib import Path
import shutil
import platform

In [2]:
FONTS_DIR = Path("fonts")

# Очищать папки перед копированием
CLEAR_EXISTING = False

# Максимум шрифтов на категорию (None = без лимита)
MAX_FONTS_PER_CATEGORY = None

## Классификация шрифтов

In [3]:
CATEGORIES = {
    "sans": [
        "arial", "roboto", "helvetica", "verdana", "tahoma",
        "calibri", "segoe", "ubuntu", "open sans", "noto sans",
        "dejavu sans", "liberation sans", "sans",
    ],
    "serif": [
        "times", "georgia", "cambria", "garamond", "baskerville",
        "palatino", "book antiqua", "serif",
        "liberation serif", "dejavu serif", "noto serif",
    ],
    "script": [
        "script", "hand", "cursive", "brush", "chancery", "comic",
        "calligraphy", "dance", "pacifico", "lobster", "caveat",
        "great vibes",
    ],
}

FONT_EXTENSIONS = {".ttf", ".otf"}

In [4]:
def get_system_font_dirs():
    system = platform.system()
    home = Path.home()
    if system == "Windows":
        return [Path("C:/Windows/Fonts"), home / "AppData/Local/Microsoft/Windows/Fonts"]
    raise RuntimeError(f"Unsupported OS: {system}")

# Возвращает уникальные системные шрифты.
def find_fonts():
    fonts = {}
    for directory in get_system_font_dirs():
        if not directory.exists():
            continue
        for path in directory.rglob("*"):
            if path.is_file() and path.suffix.lower() in FONT_EXTENSIONS:
                fonts[path.name.lower()] = path
    return list(fonts.values())

# Определяет категорию шрифта по имени файла.
def classify_font(path):
    name = path.stem.lower()
    for category in ("script", "sans", "serif"):
        if any(keyword in name for keyword in CATEGORIES[category]):
            return category
    return "other"

In [5]:
def prepare_dirs():
    for category in list(CATEGORIES) + ["other"]:
        directory = FONTS_DIR / category
        directory.mkdir(parents=True, exist_ok=True)

        if CLEAR_EXISTING:
            for file in directory.iterdir():
                if file.is_file():
                    file.unlink()

In [6]:
def copy_fonts(fonts):
    counters = {category: 0 for category in list(CATEGORIES) + ["other"]}

    for font_path in fonts:
        category = classify_font(font_path)

        if MAX_FONTS_PER_CATEGORY is not None and counters[category] >= MAX_FONTS_PER_CATEGORY:
            continue

        destination = FONTS_DIR / category / font_path.name
        if destination.exists():
            continue

        try:
            shutil.copy2(font_path, destination)
            counters[category] += 1
        except PermissionError:
            print(f"Нет доступа: {font_path}")
        except Exception as e:
            print(f"Ошибка при копировании {font_path}: {e}")

    return counters

In [7]:
prepare_dirs()

print("Поиск системных шрифтов...")
fonts = find_fonts()
print(f"Найдено шрифтов: {len(fonts)}")

counters = copy_fonts(fonts)

print(f"\nГотово. Шрифты сохранены в: {FONTS_DIR.resolve()}\n")
print("Результат:")
for category, count in counters.items():
    print(f"  {category:8s}: {count}")

Поиск системных шрифтов...
Найдено шрифтов: 375

Готово. Шрифты сохранены в: C:\Users\LordGum\Desktop\yandex cup\2026 Авито - Bootcamp\fonts

Результат:
  sans    : 37
  serif   : 11
  script  : 8
  other   : 319


# Проверка какие шрифты поддерживают русский

In [8]:
# !pip install fonttools

In [9]:
from pathlib import Path
from fontTools.ttLib import TTFont, TTLibError

REQUIRED_CHARS = (
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "abcdefghijklmnopqrstuvwxyz"
    "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"
    "абвгдеёжзийклмнопрстуфхцчшщъыьэюя"
    "0123456789"
)

In [10]:
def get_supported_codepoints(font_path):
    try:
        with TTFont(font_path, fontNumber=0, lazy=True) as font:
            cmap = font.getBestCmap()
            return set(cmap.keys()) if cmap else set()
    except (TTLibError, Exception) as e:
        print(f"  ! не удалось прочитать {font_path}: {e}")
        return None


def check_font(font_path):
    supported = get_supported_codepoints(font_path)
    if supported is None:
        return False
    return all(ord(ch) in supported for ch in REQUIRED_CHARS)


def clean_fonts(fonts_dir):
    fonts_dir = Path(fonts_dir)
    fonts = [
        p for p in fonts_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".ttf", ".otf", ".ttc"}
    ]

    good = deleted = 0

    for font in fonts:
        supported = get_supported_codepoints(font)
        if supported is None:
            print(f"DELETE (битый) {font}")
            font.unlink()
            deleted += 1
            continue

        missing = [ch for ch in REQUIRED_CHARS if ord(ch) not in supported]
        if missing:
            print(f"DELETE {font}  (нет {len(missing)} символов, напр. {''.join(missing[:8])!r})")
            font.unlink()
            deleted += 1
        else:
            print(f"OK     {font}")
            good += 1

    print()
    print(f"Всего:     {len(fonts)}")
    print(f"Оставлено: {good}")
    print(f"Удалено:   {deleted}")

In [11]:
clean_fonts(FONTS_DIR)

DELETE fonts\other\AGENCYB.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
DELETE fonts\other\AGENCYR.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
DELETE fonts\other\AKHBR.TTF  (нет 128 символов, напр. 'ABCDEFGH')
DELETE fonts\other\AKHBRBD.TTF  (нет 128 символов, напр. 'ABCDEFGH')
DELETE fonts\other\ALGER.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
OK     fonts\other\ANTQUAB.TTF
OK     fonts\other\ANTQUABI.TTF
OK     fonts\other\ANTQUAI.TTF
DELETE fonts\other\ARABSQ.TTF  (нет 128 символов, напр. 'ABCDEFGH')
DELETE fonts\other\ARBSQDTP.TTF  (нет 128 символов, напр. 'ABCDEFGH')
OK     fonts\other\ariblk.ttf
DELETE fonts\other\ARLRDBD.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
OK     fonts\other\bahnschrift.ttf
DELETE fonts\other\BASKVILL.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
DELETE fonts\other\BAUHS93.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
DELETE fonts\other\BELL.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
DELETE fonts\other\BELLB.TTF  (нет 66 символов, напр. 'АБВГДЕЁЖ')
DELETE fonts\other\BELLI.TTF 